# Lab 1: Spark Fundamentals with NYC Taxi Data

**Course:** GIK2Q3 Applied Big Data and Cloud Computing  
**Published:** Week 5

---

## Overview

In this lab, you'll apply what you learned in the Learning Lab and Lecture 2 by working with **real-world data**: the NYC Yellow Taxi Trip dataset. This dataset contains millions of taxi trips in New York City and is commonly used for big data examples because of its size and richness.

### Learning Objectives

By completing this lab, you will:
- Load and explore real-world data in Parquet format
- Understand Spark's architecture: partitions, parallelism, and data distribution
- Compare RDDs and DataFrames — understanding when to use each
- Apply core Spark operations: `select`, `filter`, `groupBy`, `agg`, `withColumn`
- Classify transformations as narrow or wide, and predict shuffle behavior
- Understand lazy evaluation through hands-on experiments
- Use the Spark UI to monitor job execution and understand stages
- Make informed decisions about caching for performance optimization
- Read and write data in Parquet format

### Grading

This lab is graded **U/G** (Pass/Fail). Each exercise has a specific answer that will be verified. Complete all exercises correctly to pass.

**Total: 12 exercises**

### Submission

Upload your completed notebook (`.ipynb` file) to the designated assignment on Canvas before the deadline.

---

## Section 1: Environment Setup

Let's set up our environment and download the data. Pay attention to the SparkSession configuration — understanding these options is important for tuning Spark applications.

In [1]:
# Check Python version
import sys
print(f"Python version: {sys.version}")

Python version: 3.13.2 (main, Feb  4 2025, 14:51:09) [Clang 16.0.0 (clang-1600.0.26.6)]


In [2]:
# Install PySpark if needed (mainly for Colab)
try:
    import pyspark
    print(f"PySpark already installed: version {pyspark.__version__}")
except ImportError:
    print("Installing PySpark...")
    %pip install pyspark -q
    import pyspark
    print(f"PySpark installed: version {pyspark.__version__}")

PySpark already installed: version 4.1.1


In [3]:
# Import libraries
import os
import time
import urllib.request
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

print("Libraries imported successfully!")

Libraries imported successfully!


In [4]:
# Environment configuration for local installations
import os

# Set JAVA_HOME for local installations (macOS/Linux)
# This is only needed for local development - Colab has Java pre-installed
java_paths = [
    "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",  # macOS Apple Silicon
    "/usr/local/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home",     # macOS Intel
    "/usr/lib/jvm/java-17-openjdk-amd64",                              # Ubuntu/Debian
]

java_home = None

for path in java_paths:
    if os.path.exists(path):
        os.environ["JAVA_HOME"] = path
        java_home = path
        break

if java_home:
    print(f"✓ JAVA_HOME set to: {java_home}")
else:
    print("⚠️ Java not found. PySpark requires Java 17.")
    print("  On macOS: brew install openjdk@17")
    print("  On Ubuntu: sudo apt install openjdk-17-jdk")

✓ JAVA_HOME set to: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home


In [5]:
# Create SparkSession - the entry point to all Spark functionality
# 
# Configuration explained:
# - appName: Identifies your application in the Spark UI and logs
# - master: "local[*]" means run locally using ALL available CPU cores
#   - "local[1]" would use only 1 core (no parallelism)
#   - "local[4]" would use exactly 4 cores
#   - In production, this would be "yarn", "k8s://...", etc.
# - spark.driver.memory: Memory allocated to the driver process (default is 1g)
#   - Increase this if you're collecting large results or have many partitions

spark = SparkSession.builder \
    .appName("Lab1-NYC-Taxi") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate() 

print(f"Spark version: {spark.version}")
print(f"Spark UI available at: {spark.sparkContext.uiWebUrl}")
print(f"Number of cores available: {spark.sparkContext.defaultParallelism}")
print("\n⚠️ Keep the Spark UI URL handy - you'll need it for exercises!")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/14 08:29:05 WARN Utils: Your hostname, MB-Pro-Merony7.local, resolves to a loopback address: 127.0.0.1; using 192.168.0.36 instead (on interface en0)
26/02/14 08:29:05 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/14 08:29:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.1.1
Spark UI available at: http://192.168.0.36:4040
Number of cores available: 8

⚠️ Keep the Spark UI URL handy - you'll need it for exercises!


### 1.1 Download NYC Taxi Data

We'll use the **January 2024** Yellow Taxi trip data from the NYC Taxi & Limousine Commission.

The data is in **Parquet format** - a columnar storage format that's highly efficient for analytical queries.

In [6]:
# Create data directory if it doesn't exist
data_dir = "data"
os.makedirs(data_dir, exist_ok=True)

# NYC Taxi data URL (January 2024)
data_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-01.parquet"
data_file = os.path.join(data_dir, "yellow_tripdata_2024-01.parquet")

# Download if not already present
if not os.path.exists(data_file):
    print(f"Downloading NYC Taxi data (~50MB)...")
    print(f"URL: {data_url}")
    urllib.request.urlretrieve(data_url, data_file)
    print(f"✓ Downloaded to {data_file}")
else:
    print(f"✓ Data already exists at {data_file}")

# Check file size
file_size_mb = os.path.getsize(data_file) / (1024 * 1024)
print(f"File size: {file_size_mb:.1f} MB")

✓ Data already exists at data/yellow_tripdata_2024-01.parquet
File size: 47.6 MB


---

## Section 2: Loading and Exploring Data (30 min)

### 2.1 Load the Parquet File

One of Spark's strengths is its ability to read various file formats. Parquet is especially efficient because:
- It's **columnar** (only reads columns you need)
- It includes **schema information** (no need to infer types)
- It supports **compression** (smaller files)

In [7]:
# Load the Parquet file
df = spark.read.parquet(data_file)

# Show schema
print("Dataset Schema:")
df.printSchema()

Dataset Schema:
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



### Understanding the Data

Key columns in the NYC Taxi dataset:

| Column | Description |
|--------|-------------|
| `tpep_pickup_datetime` | When the trip started |
| `tpep_dropoff_datetime` | When the trip ended |
| `passenger_count` | Number of passengers |
| `trip_distance` | Distance travelled (miles) |
| `PULocationID` | Pickup location zone ID |
| `DOLocationID` | Dropoff location zone ID |
| `fare_amount` | Base fare |
| `tip_amount` | Tip paid |
| `total_amount` | Total charged to customer |
| `payment_type` | 1=Credit card, 2=Cash, etc. |

In [8]:
# View first few rows
df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|2       |2024-01-01 00:57:55 |2024-01-01 01:17:43  |1              |1.72         |1         |N                 |186         |79          |2           |17.7       |1.0  |0.5    |0.0      

In [9]:
# Basic statistics
print("Dataset Statistics:")
row_count = df.count()
print(f"Total trips: {row_count:,}")

Dataset Statistics:
Total trips: 2,964,624


### 2.2 Understanding Partitions

**Partitions** are the fundamental unit of parallelism in Spark. When you load data, Spark splits it into partitions that can be processed in parallel across different CPU cores (or nodes in a cluster).

Key concepts:
- More partitions = more parallelism (up to available cores)
- Ideal partition size: 100MB - 1GB
- Too few partitions = underutilized resources
- Too many partitions = scheduling overhead

In [10]:
# Check the current number of partitions
num_partitions = df.rdd.getNumPartitions()
print(f"Number of partitions: {num_partitions}")

# See how many rows are in each partition
# glom() collects each partition into a list, then we count the length
partition_sizes = df.rdd.glom().map(len).collect()
print(f"\nRows per partition: {partition_sizes}")
print(f"Min: {min(partition_sizes):,}, Max: {max(partition_sizes):,}, Avg: {sum(partition_sizes)//len(partition_sizes):,}")

Number of partitions: 8



Rows per partition: [0, 1048576, 0, 1048576, 0, 0, 867472, 0]
Min: 0, Max: 1,048,576, Avg: 370,578


### Controlling Partitions: repartition() vs coalesce()

- **`repartition(n)`**: Redistributes data into `n` partitions. Causes a **shuffle** (data moves between nodes). Use when you need to *increase* partitions or need even distribution.

- **`coalesce(n)`**: Reduces partitions to `n` *without* a full shuffle. More efficient than repartition when *decreasing* partitions. Cannot increase partitions.

In [11]:
# Example: Repartition to 8 partitions
df_8_partitions = df.repartition(8)
print(f"After repartition(8): {df_8_partitions.rdd.getNumPartitions()} partitions")

# Example: Coalesce to 2 partitions (more efficient for reducing)
df_2_partitions = df.coalesce(2)
print(f"After coalesce(2): {df_2_partitions.rdd.getNumPartitions()} partitions")

After repartition(8): 8 partitions
After coalesce(2): 2 partitions


---

## Section 3: RDDs vs DataFrames

Spark has two main abstractions for distributed data:

| Feature | RDD (Resilient Distributed Dataset) | DataFrame |
|---------|-------------------------------------|-----------|
| **Level** | Low-level | High-level |
| **API** | Functional (map, filter, reduce) | SQL-like (select, filter, groupBy) |
| **Schema** | No schema (untyped) | Has schema (typed columns) |
| **Optimization** | None (you control everything) | Catalyst optimizer |
| **Performance** | Slower (no optimization) | Faster (optimized execution) |
| **Use case** | Custom transformations, unstructured data | Structured/semi-structured data |

**When to use RDDs:**
- Working with unstructured data (text, binary)
- Need fine-grained control over physical execution
- Custom partitioning logic

**When to use DataFrames (recommended):**
- Working with structured data (tables, JSON, Parquet)
- Standard transformations and aggregations
- Want Spark to optimize your queries

### 3.1 Creating and Using RDDs

RDDs are created from the SparkContext (`spark.sparkContext` or `sc`). Let's see the basic RDD operations.

In [12]:
# Get the SparkContext from SparkSession
sc = spark.sparkContext

# Create an RDD from a Python list using parallelize()
numbers = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
numbers_rdd = sc.parallelize(numbers)

print(f"Created RDD with {numbers_rdd.count()} elements")
print(f"Partitions: {numbers_rdd.getNumPartitions()}")

# RDD transformations use functional programming style
squared_rdd = numbers_rdd.map(lambda x: x ** 2)  # Transform each element
even_rdd = numbers_rdd.filter(lambda x: x % 2 == 0)  # Filter elements

# Actions collect results
print(f"\nOriginal: {numbers_rdd.collect()}")
print(f"Squared: {squared_rdd.collect()}")
print(f"Even only: {even_rdd.collect()}")

Created RDD with 10 elements
Partitions: 8

Original: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]
Squared: [1, 4, 9, 16, 25, 36, 49, 64, 81, 100]
Even only: [2, 4, 6, 8, 10]


### 3.2 Converting Between RDDs and DataFrames

You can convert between RDDs and DataFrames. The DataFrame's underlying data is actually an RDD!

In [13]:
# Access the underlying RDD of a DataFrame
taxi_rdd = df.rdd
print(f"Type: {type(taxi_rdd)}")
print(f"First row as RDD element: {taxi_rdd.first()}")

# Convert RDD to DataFrame (need to provide schema or use Row objects)
from pyspark.sql import Row

# Create RDD of Row objects
people_rdd = sc.parallelize([
    Row(name="Alice", age=30),
    Row(name="Bob", age=25),
    Row(name="Charlie", age=35)
])

# Convert to DataFrame
people_df = spark.createDataFrame(people_rdd)
people_df.show()

Type: <class 'pyspark.core.rdd.RDD'>


First row as RDD element: Row(VendorID=2, tpep_pickup_datetime=datetime.datetime(2024, 1, 1, 0, 57, 55), tpep_dropoff_datetime=datetime.datetime(2024, 1, 1, 1, 17, 43), passenger_count=1, trip_distance=1.72, RatecodeID=1, store_and_fwd_flag='N', PULocationID=186, DOLocationID=79, payment_type=2, fare_amount=17.7, extra=1.0, mta_tax=0.5, tip_amount=0.0, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=22.7, congestion_surcharge=2.5, Airport_fee=0.0)
+-------+---+
|   name|age|
+-------+---+
|  Alice| 30|
|    Bob| 25|
|Charlie| 35|
+-------+---+



### Exercise 1: RDD Operations (10 points)

**Task:** Using RDD operations, calculate the **sum of squares** of all numbers from 1 to 100.

Steps:
1. Create an RDD containing numbers 1 through 100 using `sc.parallelize(range(1, 101))`
2. Use `map()` to square each number
3. Use `reduce()` to sum all the squared values
4. Store the result in `answer_1`

*Note: The formula for sum of squares 1² + 2² + ... + n² = n(n+1)(2n+1)/6. For n=100, this equals 338,350.*

In [14]:
# Exercise 1: Sum of squares using RDD operations
# Store your answer in the variable answer_1

numbers_rdd = sc.parallelize(range(1, 101))  # Create RDD with numbers 1-100
squared_rdd = numbers_rdd.map(lambda x: x ** 2)  # Square each number
answer_1 = squared_rdd.reduce(lambda a, b: a + b)  # Sum all squared numbers

print(f"Answer 1 - Sum of squares (1 to 100): {answer_1:,}")

Answer 1 - Sum of squares (1 to 100): 338,350


---

## Section 4: Core DataFrame Operations

Now let's practice the fundamental DataFrame operations on the taxi data. Remember from the lecture:

- **Transformations** (lazy): `select`, `filter`, `withColumn`, `groupBy` → return new DataFrame
- **Actions** (trigger execution): `count`, `show`, `collect`, `write` → return results

### Exercise 2: Counting Trips (10 points)

**Task:** Store the total number of trips in the dataset in a variable called `answer_2`.

In [15]:
# Exercise 2: Count total trips
# Store your answer in the variable answer_2

answer_2 = df.count()  # Replace with your code

print(f"Answer 2 - Total trips: {answer_2:,}")

Answer 2 - Total trips: 2,964,624


### Exercise 3: Filtering Data (10 points)

**Task:** Count how many trips had **more than 4 passengers** (passenger_count > 4). Store the result in `answer_3`.

*Hint: Use `.filter()` followed by `.count()`*

In [16]:
# Exercise 3: Filter trips with more than 4 passengers
# Store your answer in the variable answer_3

answer_3 = df.filter(df.passenger_count > 4).count() # Replace with your code

print(f"Answer 3 - Trips with >4 passengers: {answer_3:,}")

Answer 3 - Trips with >4 passengers: 55,919


### Exercise 4: Computing Aggregates (10 points)

**Task:** Calculate the **total revenue** (sum of `total_amount` column) across all trips. Store the result in `answer_4` (rounded to 2 decimal places).

*Hint: Use `.agg(F.sum('total_amount'))` and `.collect()` to get the value*

In [17]:
# Exercise 4: Calculate total revenue
# Store your answer in the variable answer_4 (rounded to 2 decimal places)

answer_4 = df.agg(F.sum("total_amount")).collect()[0][0]
answer_4 = round(answer_4, 2)  # Round to 2 decimal places
print(f"Answer 4 - Total revenue ${answer_4:,.2f}")

# agg(): Performs aggregation
# F.sum("total_amount"): Sums the total_amount column
# collect()[0][0]: Extracts the value from the result

Answer 4 - Total revenue: $79,456,384.28


### Exercise 5: GroupBy Aggregation (10 points)

**Task:** Find the **average tip percentage** for credit card payments (payment_type = 1).

Calculate tip percentage as: `(tip_amount / fare_amount) * 100`

Only include trips where `fare_amount > 0` to avoid division by zero.

Store the average tip percentage in `answer_5` (rounded to 2 decimal places).

*Hint: Filter first, then use withColumn to create a new column, then aggregate*

In [18]:
# Exercise 5: Average tip percentage for credit card payments
# Store your answer in the variable answer_5 (rounded to 2 decimal places)

from pyspark.sql.functions import col

filtered_df = df.filter(
    (df.payment_type == 1) &
    (df.fare_amount > 0)
)

tip_df = filtered_df.withColumn(
    "tip_percentage",
    (col("tip_amount") / col("fare_amount")) * 100
)

answer_5 = round(
    tip_df.selectExpr("avg(tip_percentage)").collect()[0][0],
    2
) # Replace with your code

print(f"Answer 5 - Average tip percentage (credit card): {answer_5:.2f}%")

Answer 5 - Average tip percentage (credit card): 28.70%


### Exercise 6: Finding Top Locations (10 points)

**Task:** Find the **most popular pickup location** (the `PULocationID` with the most trips). Store the location ID in `answer_6`.

*Hint: Use groupBy, count, and orderBy(descending), then get the first row*

In [19]:
# Exercise 6: Most popular pickup location
# Store your answer in the variable answer_6 (the location ID as an integer)

answer_6 = (
    df.groupBy("PULocationID")
      .count()
      .orderBy("count", ascending=False)
      .first()[0]
) # Replace with your code

print(f"Answer 6 - Most popular pickup location ID: {answer_6}")

Answer 6 - Most popular pickup location ID: 132


---

## Section 5: Understanding Lazy Evaluation

Spark's lazy evaluation means transformations are not executed until an action is called. Let's see this in action.

### 5.1 Timing Experiment

In [20]:
# Build a chain of transformations
start = time.time()

# These are all TRANSFORMATIONS (lazy)
result = df \
    .filter(F.col("trip_distance") > 5) \
    .filter(F.col("fare_amount") > 20) \
    .withColumn("fare_per_mile", F.col("fare_amount") / F.col("trip_distance")) \
    .groupBy("PULocationID") \
    .agg(F.avg("fare_per_mile").alias("avg_fare_per_mile"))

transform_time = time.time() - start
print(f"Transformation time: {transform_time:.4f} seconds")
print("(No computation happened yet - Spark just recorded the plan!)")

Transformation time: 0.0398 seconds
(No computation happened yet - Spark just recorded the plan!)


In [21]:
# Now trigger an ACTION
start = time.time()

# This is an ACTION - triggers actual computation
output = result.orderBy(F.desc("avg_fare_per_mile")).limit(10).collect()

action_time = time.time() - start
print(f"Action time: {action_time:.4f} seconds")
print("\n(Now the work actually happened!)")

print("\nTop 10 pickup locations by average fare per mile:")
for row in output:
    print(f"  Location {row['PULocationID']}: ${row['avg_fare_per_mile']:.2f}/mile")

Action time: 0.3007 seconds

(Now the work actually happened!)

Top 10 pickup locations by average fare per mile:
  Location 111: $16.79/mile
  Location 118: $8.24/mile
  Location 8: $8.15/mile
  Location 134: $7.23/mile
  Location 157: $7.04/mile
  Location 194: $6.72/mile
  Location 93: $6.63/mile
  Location 82: $6.32/mile
  Location 1: $5.97/mile
  Location 260: $5.82/mile


### 5.2 Viewing the Execution Plan

Use `.explain()` to see how Spark plans to execute your query. This is especially useful if you can't access the Spark UI.

In [22]:
# View the logical and physical plan
result.explain(extended=True)

== Parsed Logical Plan ==
'Aggregate ['PULocationID], ['PULocationID, 'avg('fare_per_mile) AS avg_fare_per_mile#197]
+- Project [VendorID#0, tpep_pickup_datetime#1, tpep_dropoff_datetime#2, passenger_count#3L, trip_distance#4, RatecodeID#5L, store_and_fwd_flag#6, PULocationID#7, DOLocationID#8, payment_type#9L, fare_amount#10, extra#11, mta_tax#12, tip_amount#13, tolls_amount#14, improvement_surcharge#15, total_amount#16, congestion_surcharge#17, Airport_fee#18, (fare_amount#10 / trip_distance#4) AS fare_per_mile#196]
   +- Filter (fare_amount#10 > cast(20 as double))
      +- Filter (trip_distance#4 > cast(5 as double))
         +- Relation [VendorID#0,tpep_pickup_datetime#1,tpep_dropoff_datetime#2,passenger_count#3L,trip_distance#4,RatecodeID#5L,store_and_fwd_flag#6,PULocationID#7,DOLocationID#8,payment_type#9L,fare_amount#10,extra#11,mta_tax#12,tip_amount#13,tolls_amount#14,improvement_surcharge#15,total_amount#16,congestion_surcharge#17,Airport_fee#18] parquet

== Analyzed Logical 

### 5.3 Narrow vs Wide Transformations

Understanding the difference between narrow and wide transformations is crucial for Spark performance:

| Type | Description | Examples | Shuffle? |
|------|-------------|----------|----------|
| **Narrow** | Each input partition contributes to at most one output partition | `map`, `filter`, `select`, `union` | No |
| **Wide** | Input partitions contribute to multiple output partitions | `groupBy`, `join`, `orderBy`, `repartition` | Yes |

**Shuffles are expensive** because they require:
- Serializing data
- Writing to disk
- Network transfer between nodes
- Deserializing data

### Exercise 7: Transformation Classification (10 points)

**Task:** For each operation below, classify it as **"narrow"** or **"wide"**:

1. `df.filter(F.col("trip_distance") > 5)`
2. `df.groupBy("PULocationID").count()`
3. `df.select("fare_amount", "tip_amount")`
4. `df.orderBy("total_amount")`

Store your answers as a list of strings in `answer_7`.
Example format: `["narrow", "wide", "narrow", "wide"]`

In [23]:
# Exercise 7: Classify each transformation
# Store your answer as a list of 4 strings: ["narrow" or "wide", ...]

# 1. df.filter(F.col("trip_distance") > 5)      → ?
# 2. df.groupBy("PULocationID").count()         → ?
# 3. df.select("fare_amount", "tip_amount")     → ?
# 4. df.orderBy("total_amount")                 → ?

answer_7 = ["narrow", "wide", "narrow", "wide"] # Replace with your list, e.g., ["narrow", "wide", "narrow", "wide"]

print(f"Answer 7 - Transformation classifications: {answer_7}")
# Explanation:
# 1. filter() = narrow (data stays in same partitions)
# 2. groupBy() = wide (requires shuffle = data moves between partitions)
# 3. select() = narrow (only picks columns)
# 4. orderBy() = wide (requires shuffle to sort globally)

Answer 7 - Transformation classifications: ['narrow', 'wide', 'narrow', 'wide']


---

## Section 6: Spark UI Exploration

The Spark UI is your window into what Spark is actually doing. Let's explore it.

**Open the Spark UI** using the URL printed when you created the SparkSession (usually http://localhost:4040).

If you can't access the Spark UI (e.g., on Colab), use the `.explain()` output and answer based on that.

### Exercise 8: Spark UI Analysis (10 points)

Run the cell below to trigger a job, then answer the questions.

In [24]:
# Run this cell, then examine the Spark UI
trip_counts_by_hour = df \
    .withColumn("hour", F.hour("tpep_pickup_datetime")) \
    .groupBy("hour") \
    .count() \
    .orderBy("hour") \
    .collect()

print("Trips by hour of day:")
for row in trip_counts_by_hour:
    bar = "█" * (row['count'] // 50000)
    print(f"  {row['hour']:2d}:00 - {row['count']:>8,} {bar}")

Trips by hour of day:
   0:00 -   79,094 █
   1:00 -   53,627 █
   2:00 -   37,517 
   3:00 -   24,811 
   4:00 -   16,742 
   5:00 -   18,764 
   6:00 -   41,429 
   7:00 -   83,719 █
   8:00 -  117,209 ██
   9:00 -  128,970 ██
  10:00 -  138,778 ██
  11:00 -  150,542 ███
  12:00 -  164,559 ███
  13:00 -  169,903 ███
  14:00 -  182,898 ███
  15:00 -  189,359 ███
  16:00 -  190,201 ███
  17:00 -  206,257 ████
  18:00 -  212,788 ████
  19:00 -  184,032 ███
  20:00 -  159,989 ███
  21:00 -  160,888 ███
  22:00 -  143,261 ██
  23:00 -  109,287 ██


**Answer the following questions** (type your answers in the cell below):

1. How many **jobs** were created for the cell above?
2. How many **stages** were in the job?
3. Looking at the DAG visualization (or `.explain()` output), what caused the stage boundary? (Hint: think about narrow vs wide transformations)

Store your answers as:
- `answer_8a` = number of jobs (integer)
- `answer_8b` = number of stages (integer)
- `answer_8c` = what caused the stage boundary (string: "shuffle" or "narrow")

In [25]:
# Exercise 8: Spark UI Analysis
# Fill in your answers based on what you observed

answer_8a = 1   # Number of jobs (integer)
answer_8b = 2   # Number of stages (integer)
answer_8c = "shuffle"  # What caused the stage boundary: "shuffle" or "narrow"

print(f"Answer 8a - Number of jobs: {answer_8a}")
print(f"Answer 8b - Number of stages: {answer_8b}")
print(f"Answer 8c - Stage boundary cause: {answer_8c}")

Answer 8a - Number of jobs: 1
Answer 8b - Number of stages: 2
Answer 8c - Stage boundary cause: shuffle


---

## Section 7: Caching and Performance

When you need to reuse a DataFrame multiple times, caching can dramatically improve performance. But caching uses memory, so you need to make informed decisions.

### 7.1 Understanding Caching

- **`cache()`** or **`persist()`**: Stores DataFrame in memory (and/or disk) after first computation
- **`unpersist()`**: Removes DataFrame from cache, freeing memory
- Cache when: DataFrame is reused multiple times AND fits in memory
- Don't cache when: DataFrame is used once, or is too large for memory

In [26]:
# Create a filtered DataFrame that we'll use multiple times
long_trips = df.filter(F.col("trip_distance") > 10)

# First query (uncached)
start = time.time()
count1 = long_trips.count()
time1 = time.time() - start
print(f"First count (uncached): {count1:,} trips in {time1:.2f}s")

# Second query (still uncached - Spark re-reads and re-filters!)
start = time.time()
count2 = long_trips.count()
time2 = time.time() - start
print(f"Second count (uncached): {count2:,} trips in {time2:.2f}s")

First count (uncached): 228,254 trips in 0.22s
Second count (uncached): 228,254 trips in 0.12s


In [27]:
# Now let's cache the DataFrame
long_trips.cache()

# First query after caching (this populates the cache)
start = time.time()
count3 = long_trips.count()
time3 = time.time() - start
print(f"First count after .cache(): {count3:,} trips in {time3:.2f}s")
print("(This was slower because Spark read the data AND stored it in memory)")

# Second query (now reads from cache!)
start = time.time()
count4 = long_trips.count()
time4 = time.time() - start
print(f"\nSecond count (cached): {count4:,} trips in {time4:.2f}s")
print(f"Speedup from caching: {time2/time4:.1f}x faster!")

First count after .cache(): 228,254 trips in 0.61s
(This was slower because Spark read the data AND stored it in memory)

Second count (cached): 228,254 trips in 0.04s
Speedup from caching: 3.3x faster!


In [28]:
# Clean up - unpersist the cached DataFrame
long_trips.unpersist()
print("Cache cleared.")

Cache cleared.


### Exercise 9: Caching Decision (10 points)

**Task:** You have a DataFrame `expensive_df` that takes 5 seconds to compute. You need to:
- Calculate the count
- Calculate the sum of a column
- Calculate the average of a column

**Question:** Should you cache `expensive_df` before running these three operations? 

Store your answer in `answer_9` as either `"yes"` or `"no"`, and explain your reasoning in a comment.

In [1]:
# Exercise 9: Should you cache?
# Store your answer as "yes" or "no"

answer_9 = "yes" # Replace with "yes" or "no"

# Explain your reasoning in this comment:
# 
# Explanation: Since we are using the same DataFrame 3 times,
# caching can save recomputing it each time (5 seconds × 3 = 15 seconds saved)

print(f"Answer 9 - Should you cache: {answer_9}")

Answer 9 - Should you cache: yes


---

## Section 8: Writing Data

Let's practice writing data back to disk. This is an important skill for building data pipelines.

### Exercise 10: Write and Verify (10 points)

**Task:** 
1. Filter the data to only include trips from **January 1, 2024** (the first day)
2. Write the result to a Parquet file at `data/jan1_trips.parquet`
3. Read it back and count the rows
4. Store the count in `answer_10`

In [30]:
# Exercise 10: Write and verify

from pyspark.sql import functions as F

# Step 1: Filter to January 1, 2024
# Hint: Use F.to_date() to extract the date part, compare to "2024-01-01"

jan1_trips = df.filter(
    F.to_date("tpep_pickup_datetime") == "2024-01-01"
) # Replace with your filtering code

# Step 2: Write to Parquet (use mode="overwrite" to replace if exists)
output_path = "data/jan1_trips.parquet"
# Your write code here
jan1_trips.write.mode("overwrite").parquet(output_path)

# Step 3: Read it back and count
# Your read code here

answer_10 = spark.read.parquet(output_path).count() # Replace with the count

print(f"Answer 10 - Trips on January 1, 2024: {answer_10:,}")

Answer 10 - Trips on January 1, 2024: 81,013


---

## Section 9: Putting It All Together

### Exercise 11: Complete Analysis (10 points)

**Task:** Perform a complete analysis to answer this business question:

> "What is the average fare per mile for trips **longer than 2 miles** that were paid by **credit card** (payment_type = 1)?"

Requirements:
1. Filter to trips where `trip_distance > 2` AND `payment_type == 1`
2. Filter out trips where `trip_distance <= 0` (bad data)
3. Calculate fare per mile as `fare_amount / trip_distance`
4. Compute the average
5. Store the result (rounded to 2 decimal places) in `answer_11`

In [31]:
# Exercise 11: Complete analysis
# Store your answer in the variable answer_11 (rounded to 2 decimal places)

filtered_df = df.filter(
    (col("trip_distance") > 2) &
    (col("trip_distance") > 0) &
    (col("payment_type") == 1)
)

fare_per_mile_df = filtered_df.withColumn(
    "fare_per_mile",
    col("fare_amount") / col("trip_distance")
)

answer_11 = round(
    fare_per_mile_df.selectExpr("avg(fare_per_mile)").collect()[0][0],
    2
) # Replace with your code

print(f"Answer 11 - Average fare per mile (long trips, credit card): ${answer_11:.2f}")

Answer 11 - Average fare per mile (long trips, credit card): $5.64


### Exercise 12: Efficient Column Selection (10 points)

**Task:** When working with wide DataFrames (many columns), selecting only the columns you need can improve performance.

1. Use `select()` to create a new DataFrame with only these columns: `trip_distance`, `fare_amount`, `tip_amount`, `total_amount`
2. Filter to trips where `total_amount > 100`
3. Count how many "high-value" trips there are
4. Store the count in `answer_12`

In [32]:
# Exercise 12: Efficient column selection
# Store your answer in the variable answer_12

# Selecting only needed columns first 
selected = df.select("trip_distance", "fare_amount", "tip_amount", "total_amount")

# Filter
high_value = selected.filter(F.col("total_amount") > 100)

# Count
answer_12 = high_value.count()

print(f"Answer 12 - High-value trips (>$100): {answer_12:,}")

Answer 12 - High-value trips (>$100): 39,976


---

## Summary of Your Answers

Run this cell to see all your answers. Make sure none are `None` before submitting!

In [33]:
# Summary of all answers
print("="*60)
print("LAB 1 - ANSWER SUMMARY")
print("="*60)
print(f"\nExercise 1  - Sum of squares (RDD): {answer_1}")
print(f"Exercise 2  - Total trips: {answer_2}")
print(f"Exercise 3  - Trips with >4 passengers: {answer_3}")
print(f"Exercise 4  - Total revenue: {answer_4}")
print(f"Exercise 5  - Average tip % (credit card): {answer_5}")
print(f"Exercise 6  - Most popular pickup location: {answer_6}")
print(f"Exercise 7  - Transformation classifications: {answer_7}")
print(f"Exercise 8a - Number of jobs: {answer_8a}")
print(f"Exercise 8b - Number of stages: {answer_8b}")
print(f"Exercise 8c - Stage boundary cause: {answer_8c}")
print(f"Exercise 9  - Should cache: {answer_9}")
print(f"Exercise 10 - Trips on Jan 1: {answer_10}")
print(f"Exercise 11 - Avg fare/mile (long, credit): {answer_11}")
print(f"Exercise 12 - High-value trips: {answer_12}")
print("\n" + "="*60)

# Check for missing answers
answers = [answer_1, answer_2, answer_3, answer_4, answer_5, answer_6,
           answer_7, answer_8a, answer_8b, answer_8c, answer_9,
           answer_10, answer_11, answer_12]
missing = sum(1 for a in answers if a is None)

if missing > 0:
    print(f"\n⚠️ WARNING: {missing} answer(s) are still None!")
    print("Complete all exercises before submitting.")
else:
    print("\n✅ All answers provided! Ready for submission.")

LAB 1 - ANSWER SUMMARY

Exercise 1  - Sum of squares (RDD): 338350
Exercise 2  - Total trips: 2964624
Exercise 3  - Trips with >4 passengers: 55919
Exercise 4  - Total revenue: 79456384.28
Exercise 5  - Average tip % (credit card): 28.7
Exercise 6  - Most popular pickup location: 132
Exercise 7  - Transformation classifications: ['narrow', 'wide', 'narrow', 'wide']
Exercise 8a - Number of jobs: 1
Exercise 8b - Number of stages: 2
Exercise 8c - Stage boundary cause: shuffle
Exercise 9  - Should cache: no
Exercise 10 - Trips on Jan 1: 81013
Exercise 11 - Avg fare/mile (long, credit): 5.64
Exercise 12 - High-value trips: 39976


✅ All answers provided! Ready for submission.


---

## Wrap-up

### What You Learned

1. **Spark Configuration** — Understanding SparkSession options and their impact
2. **Partitions** — How Spark distributes data and controls parallelism
3. **RDDs vs DataFrames** — When to use each and how to convert between them
4. **Core Operations** — `select`, `filter`, `groupBy`, `agg`, `withColumn`, `orderBy`
5. **Transformations** — Narrow vs wide, and their impact on shuffles
6. **Lazy Evaluation** — Why transformations are not immediately executed
7. **Spark UI** — Understanding jobs, stages, and the execution DAG
8. **Caching** — When and why to cache DataFrames
9. **File I/O** — Reading and writing Parquet files

### Key Takeaways

- **Partitions = Parallelism**: More partitions means more parallel tasks (up to available cores)
- **DataFrames > RDDs** for most use cases: Catalyst optimizer makes them faster
- **Narrow transformations** don't shuffle; **wide transformations** do
- **Shuffles are expensive**: Minimize them when possible
- **Cache wisely**: Only cache DataFrames that are reused multiple times
- **Parquet is preferred**: Columnar format with schema and compression

### Submission

Upload your completed notebook to the designated assignment on **Canvas** before the deadline.

---

In [ ]:
# Clean up - stop the Spark session
spark.stop()
print("Spark session stopped. Lab complete!")